# 01 · PaHaW exploration
Goal: inventory subjects/tasks, labels (PD/H), QC on .svc, confirm tasks 1–2–3.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# notebook is in py/notebooks/ → repo root is parents[2]
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parents[1]
elif (REPO / "py").exists():
    pass
else:
    REPO = Path("..").resolve()  # adjust if needed

sys.path.insert(0, str(REPO / "py"))
from load_svc import load_svc, parse_svc_stem

PUBLIC = REPO / "PaHaW" / "PaHaW" / "PaHaW_public"
CORPUS = REPO / "PaHaW" / "PaHaW" / "PaHaW_files" / "corpus_PaHaW.xlsx"
SELECTED_TASKS = [1, 2, 3]  # spiral, repeated l, bigram le

print(REPO)
print(PUBLIC.exists(), CORPUS.exists())

c:\Users\frank\Documents\Uni\SecureCloudComputing\Parkinson’s Disease Classification\Parkinson-classifier-saas-architecture
True True


In [2]:
files = list(PUBLIC.glob("*/*.svc"))
subjects = sorted({p.parent.name for p in files})
print(f"subjects={len(subjects)}  svc_files={len(files)}")

rows = []
for p in files:
    sid, task, rep = parse_svc_stem(p.stem)
    rows.append({"subject": sid, "task": int(task), "rep": rep, "path": p})
inv = pd.DataFrame(rows)
print(inv.groupby("task")["subject"].nunique())
print("subjects missing task 1:", sorted(set(subjects) - set(inv.loc[inv.task == 1, "subject"])))

subjects=75  svc_files=597
task
1    72
2    75
3    75
4    75
5    75
6    75
7    75
8    75
Name: subject, dtype: int64
subjects missing task 1: ['00061', '00080', '00089']


In [3]:
meta = pd.read_excel(CORPUS)
print(meta.columns.tolist())
print(meta.head())

# adatta il nome colonna ID se diverso
id_col = "ID" if "ID" in meta.columns else meta.columns[0]
y_col = "Disease"
print(meta[y_col].value_counts(dropna=False))
print(meta[["PD status"]].value_counts(dropna=False) if "PD status" in meta.columns else "no PD status")

['ID', 'Nationality', 'Sex', 'Disease', 'PD status', 'Age', 'Dominant hand', 'LED', 'UPDRS III', 'Length of PD']
   ID Nationality Sex Disease PD status  Age Dominant hand     LED  UPDRS III  \
0   1       Czech   F      PD        ON   68             R  1115.0        2.0   
1   2       Czech   F      PD        ON   78             R  2110.0        2.0   
2   3       Czech   F      PD        ON   69             R  1556.6        2.0   
3   4       Czech   F      PD        ON   79             R  1691.0        2.0   
4   5       Czech   F      PD        ON   69             R   600.0        2.0   

   Length of PD  
0           6.0  
1           8.0  
2           7.0  
3          12.0  
4           2.0  
Disease
H     38
PD    37
Name: count, dtype: int64
PD status
nan          38
ON           37
Name: count, dtype: int64


In [5]:
sample = PUBLIC / "00001" / "00001__1_1.svc"
df = load_svc(sample)
print(sample.name, df.shape)
print(df.head())
print(df["button"].value_counts())
print(df[["x", "y", "pressure"]].describe())

00001__1_1.svc (1772, 7)
      y     x       t  button  azimuth  altitude  pressure
0  4161  2474  625877       1     3447       633        78
1  4164  2472  625884       1     3457       624       124
2  4166  2472  625892       1     3457       624       162
3  4167  2471  625899       1     3457       624       212
4  4168  2471  625907       1     3457       624       266
button
1    1772
Name: count, dtype: int64
                 x            y     pressure
count  1772.000000  1772.000000  1772.000000
mean   2496.901242  4236.053047   850.491535
std     760.467751   685.738527   192.388956
min    1044.000000  2848.000000    58.000000
25%    1952.750000  3740.750000   743.000000
50%    2469.000000  4234.500000   907.000000
75%    3060.250000  4772.250000   987.000000
max    4046.000000  5523.000000  1219.000000


In [6]:
qc = []
for p in files:
    sid, task, rep = parse_svc_stem(p.stem)
    d = load_svc(p, warn_mismatch=True)
    qc.append({
        "subject": sid,
        "task": int(task),
        "n": len(d),
        "on_surface_pct": (d["button"] == 1).mean(),
        "duration_ms": int(d["t"].max() - d["t"].min()) if len(d) else None,
    })
qc = pd.DataFrame(qc)
print(qc.groupby("task")["n"].describe()[["count", "mean", "min", "max"]])
print("empty files:", qc.loc[qc.n == 0])

warn 00005__2_1.svc: header=1396, rows=1344
warn 00010__2_1.svc: header=2114, rows=2111
warn 00051__5_1.svc: header=1870, rows=1620
warn 00075__8_1.svc: header=1663, rows=1646
warn 00095__5_1.svc: header=4115, rows=2887
      count         mean     min      max
task                                     
1      72.0  2873.694444   552.0  16071.0
2      75.0  1668.013333   405.0   4226.0
3      75.0  1984.160000   693.0   6615.0
4      75.0  2305.013333   614.0   6827.0
5      75.0  2608.480000   855.0   7993.0
6      75.0  2315.080000   796.0   5783.0
7      75.0  1469.293333   585.0   4423.0
8      75.0  3086.133333  1234.0   7676.0
empty files: Empty DataFrame
Columns: [subject, task, n, on_surface_pct, duration_ms]
Index: []
